# Notebook 03 — Automatic Clustering
## Best Case vs. Worst Case

**Key Insight**: When your queries don't align with natural load order, a cluster key reorganizes partitions to match your access pattern.

| | Best Design | Worst Design |
|---|---|---|
| **Cluster key** | `account_id` (matches customer lookup pattern) | `currency_code` (only 3 values — useless) |
| **After reclustering** | 95%+ partition pruning | Zero improvement, wasted credits |

In [ ]:
USE ROLE SYSADMIN;
USE WAREHOUSE HOL_XS;
USE SCHEMA JPMC_PERF_HOL.CONSUMER_BANKING;

-- Disable result cache for accurate performance comparisons
ALTER SESSION SET USE_CACHED_RESULT = FALSE;

---
## Step 1: Baseline — The Problem

The `TRANSACTIONS_UNORDERED` table has randomly shuffled data.  
Customer account lookups must scan nearly ALL partitions because `account_id` is scattered everywhere.

In [ ]:
-- Check clustering quality on the UNORDERED table (should be POOR)
SELECT SYSTEM$CLUSTERING_INFORMATION('TRANSACTIONS_UNORDERED', '(account_id)');

In [ ]:
-- WORST CASE: account_id lookup on shuffled table (full scan)
SELECT transaction_date, merchant_name, amount, channel
FROM TRANSACTIONS_UNORDERED
WHERE account_id = 'ACC-0000012345'
ORDER BY transaction_date DESC
LIMIT 50;

**Check Query Profile** → Notice: partitions scanned ≈ total partitions (full scan)

---
## Step 2: Best Case — Create Clustered Table on `account_id`

We create a new copy ordered by `account_id` with a cluster key applied.

In [ ]:
-- Use XL for creating the 500M-row clustered copy
USE WAREHOUSE HOL_XL;

-- Create a clustered copy (ordered by account_id during CTAS)
CREATE OR REPLACE TABLE TRANSACTIONS_CLUSTERED
    CLUSTER BY (account_id)
AS
SELECT * FROM TRANSACTIONS_UNORDERED
ORDER BY account_id;

In [ ]:
-- Switch back to XS for performance testing
USE WAREHOUSE HOL_XS;

In [ ]:
-- Verify clustering quality (should be excellent — average_depth close to 1)
SELECT SYSTEM$CLUSTERING_INFORMATION('TRANSACTIONS_CLUSTERED', '(account_id)');

In [ ]:
-- BEST CASE: Same query on clustered table (massive pruning)
SELECT transaction_date, merchant_name, amount, channel
FROM TRANSACTIONS_CLUSTERED
WHERE account_id = 'ACC-0000012345'
ORDER BY transaction_date DESC
LIMIT 50;

**Open the Query Profile** → Compare:
- Unordered table: ~90-100% partitions scanned
- Clustered table: <5% partitions scanned
- Execution time: dramatic improvement

---
## Step 3: Worst Case — Clustering on a Useless Column

What happens if you cluster on `currency_code`? (only 3 distinct values: USD, GBP, EUR)

In [ ]:
-- Check: currency_code is ALREADY well-clustered (low cardinality = few values per partition anyway)
SELECT SYSTEM$CLUSTERING_INFORMATION('TRANSACTIONS_UNORDERED', '(currency_code)');

**What you'll see**: The table is already well-clustered on `currency_code` because with only 3 values, most partitions naturally contain a limited range. Adding a cluster key here burns credits for zero improvement.

---
## Step 4: Side-by-Side Metrics

In [ ]:
-- Compare the two account_id queries
SELECT
    CASE WHEN query_text ILIKE '%TRANSACTIONS_CLUSTERED%' THEN 'CLUSTERED' ELSE 'UNORDERED (full scan)' END AS version,
    total_elapsed_time / 1000 AS elapsed_sec,
    query_id,
    bytes_scanned / (1024*1024*1024) AS gb_scanned
FROM TABLE(SNOWFLAKE.INFORMATION_SCHEMA.QUERY_HISTORY(
    END_TIME_RANGE_START => DATEADD(minute, -15, CURRENT_TIMESTAMP()),
    END_TIME_RANGE_END => CURRENT_TIMESTAMP(),
    RESULT_LIMIT => 50
))
WHERE query_text ILIKE '%ACC-0000012345%'
  AND query_type = 'SELECT'
  AND query_text NOT ILIKE '%QUERY_HISTORY%'
ORDER BY start_time DESC
LIMIT 2;

---
## Step 5: Cost Estimation

In [ ]:
-- What would it cost to maintain clustering on account_id?
SELECT SYSTEM$ESTIMATE_AUTOMATIC_CLUSTERING_COSTS('TRANSACTIONS_UNORDERED', '(account_id)');

---
## Key Takeaways

| | Best Design | Worst Design |
|---|---|---|
| **Cluster key** | Column you frequently filter on with high cardinality (`account_id`) | Column with 3 values (`currency_code`) |
| **Result** | 95%+ pruning, 10-20x faster | Zero improvement, wasted reclustering credits |
| **When to use** | Queries don't align with natural load order | NEVER cluster on low-cardinality columns |

**Decision framework**:
1. Check natural clustering first (`SYSTEM$CLUSTERING_INFORMATION`)
2. If already good → don't add a cluster key (free performance)
3. If poor AND you query that column frequently → add cluster key
4. Estimate cost before committing (`SYSTEM$ESTIMATE_AUTOMATIC_CLUSTERING_COSTS`)

**Next** → [Notebook 04 — Warehouse Optimization](./Notebook_04_Warehouse_Optimization.ipynb)